<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🔎 M1 · 06 Lab：自动深度研究 Crew

**一句话**：4 个 agent 按「计划 → 搜集 → 核查 → 写报告」接力，把一个研究问题变成带引用的报告。

<span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">🏷️ Lab</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">📄 exercise.ipynb</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">🧩 前置：04 Hello World · 05 规划多 agent 系统</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">✅ 参考答案：solution.ipynb</span>

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**要解决的问题**：你是研究顾问，客户主题五花八门、时间又紧。需要一个**自动深度研究**方案：快速从全网搜集、核实、综合信息，交付**经过事实核查**的可靠报告。

**本 lab 学会**：给 agent 定角色、给 agent 配工具、自己写 task、把 agent + task 组成 Crew。
**课程要求**：把 `None` 换成自己的代码（`### START CODE HERE ###` 之间）。下面所有 TODO 都已是你填好的答案。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧠 第 0 步：先拆问题——谁做哪件事？

</div>

先自己想：这个 crew 要「根据用户问题做研究计划 → 执行计划 → 审核与核查事实 → 用收集到的信息写报告」，每一步该由什么「人」来做？

<div style="background:#111827; border:1px dashed #334155; border-radius:8px; padding:10px 16px; width:97%;"><div style="color:#94a3b8; font-weight:700; margin-bottom:6px;">4 个 agent ↔ 4 个 task，一对一、按顺序接力</div><span style="display:inline-block; background:#0f172a; border:1.5px solid #a78bfa; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">📝 Research Planner<br>→ 研究计划</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #60a5fa; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">🌐 Internet Researcher<br>→ 搜集资料（🔧 Exa + Scrape）</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #fbbf24; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">🕵️ Fact Checker<br>→ 核查质量（🔧 Exa + Scrape）</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #4ade80; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">✍️ Report Writer<br>→ 最终报告</span></div>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🆕 原图缺失**：本文件夹 `images/agents-tasks-diagram.png` 与 07 lab 的图**完全相同**（md5 一致，是代码审查 crew 的图），不是本 lab 的图；原 notebook 还用了绝对路径引用它。上面的流程条按课程文字重画了本 lab 的结构。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 1 步：环境准备

</div>

In [1]:
from crewai import Agent, Task, Crew
import os
os.environ["CREWAI_TESTING"] = "true"   # 跳过首次运行的 tracing 交互询问
from utils import get_openai_api_key

# set the OpenAI model (gpt-4o-mini)
os.environ["MODEL"] = "gpt-4o-mini"   # agent 没写 llm= 时的默认模型（未设则 gpt-4.1-mini）
# set up the OpenAI API key 
os.environ["OPENAI_API_KEY"] = get_openai_api_key()

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 2 步：创建工具实例（TODO）

</div>

<div style="width:97%;"><div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(230px,1fr)); gap:10px;"><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #60a5fa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🔍 EXA Search Web Loader</div><div style="font-size:0.92em; line-height:1.6;"><b>语义搜索</b>：用 exa.ai API 按「意思」而非关键词找最相关网页，比传统 embedding 检索更能捕捉概念间的关系。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #4ade80; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">📄 ScrapeWebsiteTool</div><div style="font-size:0.92em; line-height:1.6;"><b>抓网页</b>：读取指定 URL 的正文内容。搜索给链接，抓取给全文。</div></div></div></div>

这两个工具会给 **Internet Researcher**（以及 Fact Checker）用：先搜，再抓原文。

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🆕 2026-10 版本差异（crewai_tools 1.15.22）**：`EXASearchTool` 已改名 `ExaSearchTool`，旧名保留为**弃用别名**（实例化时发 `DeprecationWarning`），课程代码照常能跑。`base_url` 默认就读 `EXA_BASE_URL`（课程平台的代理地址）；本地没设这个变量 → `None` → 直连官方 API。

</div>

In [2]:
# import the tools
from crewai_tools import EXASearchTool, ScrapeWebsiteTool
from utils import get_exa_api_key

# set the exa API key
os.environ["EXA_API_KEY"] = get_exa_api_key()   # .env 缺 key → TypeError（见坑表）

### START CODE HERE ###

# Create the EXASearchTool instance
exa_search_tool = EXASearchTool(base_url=os.getenv("EXA_BASE_URL"))   # 你的答案
# Create the ScrapeWebsiteTool instance
scrape_website_tool = ScrapeWebsiteTool()   # 你的答案：无需参数

### END CODE HERE ###

/var/folders/xx/ypnl5f2n0y7b48w_pgxyhqt80000gn/T/ipykernel_37348/4161849828.py:11: DeprecationWarning: EXASearchTool is deprecated and will be removed in a future release; use ExaSearchTool instead.
  exa_search_tool = EXASearchTool(base_url=os.getenv("EXA_BASE_URL"))   # 你的答案


<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 3 步：Research Planner agent（已给出）

</div>

三要素复习：`role` = 如果是人，职位叫什么；`goal` = 这个 agent 要达成的**具体**目标；`backstory` = 突出与职责相关的技能，多用能真正提升结果的关键词。

lab 比视频多了两个参数，作用是让 lab 跑得快一点：

| 参数 | 作用 |
|---|---|
| `max_rpm` | 每分钟最多请求数，防触发限流 |
| `max_iter` | 最多迭代（思考-行动）轮数，到了就必须给出当前最佳答案 |

In [3]:
# define the research planner agent
research_planner = Agent(
    role="Research Planner",
    goal="Analyze queries and break them down into smaller, specific research topics.",
    backstory=(
         "You are a research strategist who excels at breaking down complex questions "
         "into manageable research components. You identify what needs to be researched "
         "and create clear research objectives."
    ),
    verbose=True, # set to True to see detailed agent actions
    max_rpm=150,   # 每分钟最多 150 次 LLM 请求
    max_iter=15   # 最多 15 轮就得交答案
)

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 4 步：其余 3 个 agent（TODO：backstory + tools）

</div>

`role` / `goal` 已给，自己写 `backstory`；**Internet Researcher** 和 **Fact Checker** 要通过 `tools=` 拿到工具。你的 backstory 都强调了「不编造、可追溯来源」——这正是对付幻觉的关键词。

In [4]:
researcher = Agent(
    role="Internet Researcher",
    goal="Research thoroughly all assigned topics",
    ### START CODE HERE ###
    backstory=(
        "You are a skilled researcher with experience in online investigation "
        "and data collection. You know how to find reliable sources, extract relevant information, "
        "and always verify facts across multiple sources to avoid misinformation or hallucination. "
        "You never invent facts and always trace information to its origin."
    ),
    # add the 2 tool instances you created
    tools=[exa_search_tool, scrape_website_tool],   # 研究员：先搜再抓
    ### END CODE HERE ###
    verbose=True,
    max_rpm=150,
    max_iter=15
)

fact_checker = Agent(
    role="Fact Checker",
    goal=(
        "Verify data for accuracy, identify inconsistencies, "
        "and flag potential misinformation"
    ),
    ### START CODE HERE ###
    backstory=(
        "You are a quality assurance specialist with expertise in fact-checking "
        "and identifying misinformation and hallucinations. You cross-reference information, "
        "spot inconsistencies, and ensure all data meets high accuracy standards. You rigorously"
        "check for hallucinated or invented content and require that all facts be supported"
        "by evidence."   # ⚠️ 上两行末尾缺空格 → "rigorouslycheck"
    ),
    tools=[exa_search_tool, scrape_website_tool],
    ### END CODE HERE ###
    verbose=True,
    max_rpm=150,
    max_iter=15
)

report_writer = Agent(
    role="Report Writer",
    goal="Write clear, concise, and well-structured reports based on gathered information",
    ### START CODE HERE ###
    backstory=(
         "You are an expert writer who specializes in creating clear, well-structured "
        "research reports. You synthesize complex information into readable formats and "
        "always include proper citations and sources."
    ),
    ### END CODE HERE ###
    verbose=True,max_rpm=150,
    max_iter=15
)
        

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 5 步：4 个 task

</div>

| Task | 做什么 | 产出 | agent |
|---|---|---|---|
| Create research plan | 把用户问题拆成具体主题与关键问题 | 研究主题 + 每个主题的关键问题 + 成功标准 | Research Planner |
| Gather research data | 按计划搜集所有主题的信息，引用所有来源 | 每个主题的资料 + 引用及来源可信度说明 | Internet Researcher |
| Verify information quality | 审核全部资料，找出矛盾、可能的错误信息、缺口 | 全部资料 + 审核结果（一致性检查、来源可靠性评级） | Fact Checker |
| Write final report | 用核实过的资料回答原问题；分节、带引用、给可执行建议 | 最终报告：执行摘要 + 完整引用 | Report Writer |

每个 `Task` 要定义：`description`（可分条写）、`expected_output`（要结构就写具体，比如指定 key 的字典）、`agent`（匹配已定义的 agent）。
本 lab 唯一的输入是用户问题 `user_query`，通过 `{user_query}` 占位符传进 description。

In [5]:
# define the create research plan task
create_research_plan_task = Task(
    description=(
        "Based on the user's query, break it down into specific topics and key questions, "
        "and create a focused research plan."
        "The user's query is: {user_query}"   # 普通字符串，不是 f-string！kickoff 时才替换
    ),
    expected_output=(
        "A research plan with main research topics to investigate, "
        "key questions for each topic, and success criteria for the research."
        ),
    agent=research_planner,
)

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**📝 我的笔记 —— `{user_query}` 不是 f-string，那它怎么被替换？**

`description` 是普通字符串 `"..."`，Python 这一步**什么都不做**，`{user_query}` 只是存进 `task.description` 的字面文本。

真正替换的是 **CrewAI**，在你调用 `crew.kickoff(inputs={"user_query": user_query})` 时：它遍历每个 task 的 `description` / `expected_output`（以及 agent 的 `role` / `goal` / `backstory`），做一次类似 `str.format(**inputs)` 的模板插值，把 `{user_query}` 换成 `inputs["user_query"]`。

- 语法像 `str.format()`，但替换是 CrewAI 库代码手写的，**不是**解释器做的；
- 这样设计是因为定义 `Task(...)` 时 `user_query` 可能还没赋值——**先定义模板，运行时再灌数据**，和 LangChain 的 `PromptTemplate` 思路一样。

> 🆕 2026-10 核对源码（crewai 1.15.22）：入口是 `Crew._interpolate_inputs()` → 每个 task 调 `Task.interpolate_inputs_and_add_conversation_history()`、每个 agent 调 `agent.interpolate_inputs()`（旧版叫 `Task.interpolate_inputs`）。

</div>

其余 3 个 task：`description` 已给，你填了 `expected_output` 和 `agent`。

In [6]:
# define the gather research data task
gather_research_data_task = Task(
    description=(
        "Using the research plan, collect information on all identified topics. "
        "Cite all sources used."
    ),
    ### START CODE HERE ###
    expected_output=(
        "Comprehensive research data including: information for each"   # ⚠️ 末尾缺空格 → "eachresearch"
        "research topic, and citations used along with source credibility notes"
    ),
    agent=researcher,   # 搜集 → 研究员
    ### END CODE HERE ###
)

#define the verify information quality task
verify_information_quality_task = Task(
    description=(
        "Review all collected research. Identify any conflicting information, "
        "potential misinformation, or gaps that need addressing."
    ),
    ### START CODE HERE ###
    expected_output=(
        "A report with all the original data you got plus any "
        "verified facts vs. questionable information, make sure this is as comprehensive "
        "as possible for final report generation"
    ),
    agent=fact_checker,   # 核查 → 事实核查员
    ### END CODE HERE ###
)

# define the write final report task
write_final_report_task = Task(
    description=(
        "Create a comprehensive report that answers the original query using all verified research data. "
        "Structure it with clear sections, include citations, and provide actionable insights."
    ),
    ### START CODE HERE ###
    expected_output=(
        "A final research report containing: executive summary, detailed "
        "findings that answer the user query, supporting evidence and analysis, complete "
        "source citations."
    ),
    agent=report_writer,   # 写作 → 报告撰写人
    ### END CODE HERE ###
)
    

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 6 步：组 Crew（TODO）

</div>

`agents` = 团队名单；`tasks` = **按执行顺序**排列的任务列表。

In [7]:
# create the crew with the defined agents and tasks
crew = Crew(
    ### START CODE HERE ###
    agents=[research_planner, researcher, fact_checker, report_writer],
    tasks=[create_research_plan_task, gather_research_data_task, verify_information_quality_task, write_final_report_task],   # 顺序就是执行顺序：计划→搜集→核查→写作
    ### 
)

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**📝 我的笔记 —— 上一个 task 的输出怎么「喂」给下一个？**

`Process.sequential`（默认）按 `tasks` 顺序执行，每跑完一个就把 `TaskOutput` 存进内部列表 `task_outputs`。跑下一个 task **之前**先算它的 context：`context = self._get_context(task, task_outputs)`。

| `Task(...)` 怎么写 | 拿到的 context |
|---|---|
| 不写 `context=`（本 lab 所有 task） | **之前所有** task 的 `.raw`，用 `"\n\n----------\n\n".join(...)` 拼起来 |
| `context=[task_a, task_b]` | 只拼指定 task 的 `.raw` |
| `context=[]` | 空字符串，不给上文 |

这段文本再被拼进下一个 task 的 prompt（`agent/utils.py::format_task_with_context`）：

```
<task 自己的 description + expected_output>

This is the context you're working with:
<前面 task 的 raw 输出拼接文本>
```

**关键**：不是向量检索、也不是结构化传参，纯粹是**把前面 task 的文本输出原样拼进后一个 task 的 prompt**，让下一个 agent 的 LLM「读到」。所以 `gather_research_data_task` 能看到研究计划，不用你写任何传参代码。`agents=[...]` 和这套传递无关——它只是名单 / hierarchical 模式下的调度池。

> 🆕 2026-10 已在 crewai 1.15.22 源码核对：`Crew._get_context`、`aggregate_raw_outputs_from_task_outputs`（分隔符 `\n\n----------\n\n`）、模板 `"{task}\n\nThis is the context you're working with:\n{context}"` 均与上面一致。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 7 步：写研究问题（TODO）并运行

</div>

In [8]:
### START CODE HERE ###

# Write your query, which will be used as input for the tasks.
user_query = "Evaluate the top five emerging AI tools for automating competitive market analysis, including their features, limitations, costs, and ideal use cases for a mid-sized marketing firm."   # ✏️ 换成任何你想研究的主题

### END CODE HERE ###

所有 agent 都设了 `verbose=True`，运行时能看到每个 agent 的思考、工具调用和每个 task 的输出。

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**📝 我的笔记 —— 在 Jupyter 里跑 `crew.kickoff()` 会报 `RuntimeError`**

```
RuntimeError: Agent execution was invoked synchronously from within a running
event loop. Use `agent.kickoff_async()` / `crew.kickoff_async()`
(or `await agent.aexecute_task(...)`) when calling from async code.
```

**原因**：Jupyter 的 `ipykernel` 本身一直跑着 asyncio 事件循环；这个版本的 `crewai` 内部走异步执行路径，检测到「已处于运行中的事件循环」就拒绝同步调用。纯 `.py` 脚本没有预先存在的事件循环，所以同样代码在脚本里能跑，只有 notebook 会炸。

**修复**：改成 `await crew.kickoff_async(inputs=...)`（Jupyter 支持 cell 顶层 `await`）——下面的 cell 已改。原笔记提到的另一招「开头 `nest_asyncio.apply()` 保留同步写法」见下方 🆕。

</div>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🆕 2026-10 版本差异（crewai 1.15.22，已实测）**：课程原代码是 `result = crew.kickoff(inputs=...)`，在运行中的事件循环里必报上面的错，所以这里改成 `await crew.kickoff_async(...)`；它内部是 `asyncio.to_thread(self.kickoff, ...)`，返回值同样是 `CrewOutput`。
`nest_asyncio` 在这个版本大概率**无效**：crewai 用 `asyncio.get_running_loop()` 做检测，打补丁后循环依旧「在运行」（按源码推断，未实测）。

</div>

In [9]:
result = await crew.kickoff_async(   # 课程原写法 crew.kickoff(...)，Jupyter 里会 RuntimeError（见上）
    inputs={
        "user_query": user_query,   # 填进所有 {user_query} 占位符
    }
)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Task: Based on the user's query, break it down into specific topics and key questions, and create a focused    │
│  research plan.The user's query is: Evaluate the top five emerging AI tools for automating competitive market   │
│  analysis, including their features, limitations, costs, and ideal use cases for a mid-sized marketing firm.    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Research Plan: Evaluating the Top Five Emerging AI Tools for Automating Competitive Market Analysis        │
│                                                                                                                 │
│  **Main Research Topics:**                                                                                      │
│                                                                                                                 │
│  1. **Identification of Top Five AI Tools**                                                                     │
│     - **Key Questions:**                                                                                        │
│       - What are the latest emerging AI tools specifically designed for market analysis?                        │
│       - How are these tools ranked, and what criteria are used for ranking?                                     │
│       - Which tools have garnered the most attention or popularity in the marketing industry?                   │
│     - **Success Criteria:**                                                                                     │
│       - A list of five AI tools that are recognized as emerging in the market analysis space with supporting    │
│  evidence (e.g., reviews, market traction, and user testimonials).                                              │
│                                                                                                                 │
│  2. **Features of Each Tool**                                                                                   │
│     - **Key Questions:**                                                                                        │
│       - What unique features does each AI tool offer that facilitate competitive market analysis?               │
│       - How do these features enhance the capabilities of a mid-sized marketing firm?                           │
│       - Are there any integration capabilities with existing marketing platforms (e.g., CRM, marketing          │
│  automation)?                                                                                                   │
│     - **Success Criteria:**                                                                                     │
│       - A comparative table detailing the features of each tool, including user interface, data sources,        │
│  automation capabilities, and reporting functionalities.                                                        │
│                                                                                                                 │
│  3. **Limitations of Each Tool**                                                                                │
│     - **Key Questions:**                                                                                        │
│       - What are the potential drawbacks or limitations of using each tool for market analysis?                 │
│       - Are there any technical requirements or user skills necessary to operate these tools effectively?       │
│       - What are common user complaints or issues reported for each tool?                                       │
│     - **Success Criteria:**                                                                                     │
│       - A summary of limitations for each tool, includi

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internet Researcher                                                                                     │
│                                                                                                                 │
│  Task: Using the research plan, collect information on all identified topics. Cite all sources used.            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool exa_search_tool executed with result: Title: A roundup of the latest AI-powered marketing technology releases
URL: https://martech.org/a-roundup-of-the-latest-ai-powered-marketing-technology-releases/
ID: https://martech.org/a-roundup-of-...


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact Checker                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Comprehensive Research Report on Top Emerging AI Tools for Automating Competitive Market Analysis          │
│                                                                                                                 │
│  #### 1. Identification of Top Five AI Tools                                                                    │
│                                                                                                                 │
│  Based on extensive research, the top five emerging AI tools for competitive market analysis have been          │
│  identified as follows:                                                                                         │
│                                                                                                                 │
│  1. **DeepSights by Market Logic**                                                                              │
│     - **Description**: An AI assistant that provides insights into business intelligence questions, embedding   │
│  into tools like Microsoft Teams.                                                                               │
│     - **Popularity**: Cited as widely trusted by 100,000+ users globally.                                       │
│     - **Source**: [Market Logic Press Release                                                                   │
│  2023](https://marketlogicsoftware.com/news/2023/deepsights-press-release/)                                     │
│                                                                                                                 │
│  2. **Course5 Compete**                                                                                         │
│     - **Description**: Utilizes OpenAI technology for market intelligence, focusing on improved product         │
│  matching.                                                                                                      │
│     - **Prominence**: Strong focus on creative applications with past funding of $53 million reflected by       │
│  industry recognition.                                                                                          │
│     - **Source**: [PR Newswire Announcement                                                                     │
│  2023](https://www.prnewswire.com/news-releases/course5-intelligence-integrates-openais-gpt-models-with-their-  │
│  enterprise-analytics-platforms-301777256.html)                                                                 │
│                                                                                                                 │
│  3. **Generate by Nextatlas**                                                                                   │
│     - **Description**: Combines LLMs with trend forecasting for actionable insights from social media data.     │
│     - **Recognition**: Positioning itself as a leader in capturing cultural shifts years ahead of mainstream    │
│  markets.                                                                                                       │
│     - **Source**: [Nextatlas Blog on Generate                                                                   │
│  Launch](https://blog.nextatlas.com/article/Nextalas_Generate_Trend_Prediction_Launch)                          │
│                                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Task: Create a comprehensive report that answers the original query using all verified research data.          │
│  Structure it with clear sections, include citations, and provide actionable insights.                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Comprehensive Research Report on Top Emerging AI Tools for Automating Competitive Market Analysis            │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│                                                                                                                 │
│  The landscape of competitive market analysis is increasingly influenced by artificial intelligence (AI),       │
│  enabling firms to access and interpret vast amounts of data quickly and effectively. This report identifies    │
│  and evaluates five emerging AI tools that are reshaping the market analysis field. It encompasses detailed     │
│  findings on tool features, limitations, costs, and practical use cases tailored for mid-sized marketing        │
│  firms. The objective is to equip organizations with the knowledge necessary to select appropriate AI tools     │
│  that can enhance their market analysis capabilities.                                                           │
│                                                                                                                 │
│  ## 1. Identification of Top Five AI Tools                                                                      │
│                                                                                                                 │
│  After extensive research on emerging AI technologies, the following tools have been identified as the top      │
│  five for competitive market analysis:                                                                          │
│                                                                                                                 │
│  1. **DeepSights by Market Logic**                                                                              │
│     - **Description**: An AI assistant offering natural language responses to business intelligence inquiries,  │
│  integrated with platforms like Microsoft Teams.                                                                │
│     - **Popularity**: Over 100,000 users globally.                                                              │
│     - **Source**: [Market Logic Press Release                                                                   │
│  2023](https://marketlogicsoftware.com/news/2023/deepsights-press-release/)                                     │
│                                                                                                                 │
│  2. **Course5 Compete**                                                                                         │
│     - **Description**: Leverages OpenAI technology for greater market intelligence and improved product         │
│  matching.                                                                                                      │
│     - **Prominence**: Noted for creativity and supported by a substantial funding of $53 million.               │
│     - **Source**: [PR Newswire Announcement                                                                     │
│  2023](https://www.prnewswire.com/news-releases/course5-intelligence-integrates-openais-gpt-models-with-their-  │
│  enterprise-analytics-platforms-301777256.html)        

对照上面每个 task 的输出：是否符合你的预期？不符合就回去改 `expected_output`。

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**看日志时会遇到**：Fact Checker 有时把 `include_domains` 传成字符串 `'None'` → 工具报 `Arguments validation failed ... Input should be a valid list`。这不会让 crew 崩溃：错误作为工具结果返回给 agent，它会换参数重试（旧输出里出现过 7 次）。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 8 步：渲染最终报告

</div>

In [10]:
from IPython.display import Markdown
Markdown(result.raw)    # result.raw = 最后一个 task（写报告）的输出

# Comprehensive Research Report on Top Emerging AI Tools for Automating Competitive Market Analysis

## Executive Summary

The landscape of competitive market analysis is increasingly influenced by artificial intelligence (AI), enabling firms to access and interpret vast amounts of data quickly and effectively. This report identifies and evaluates five emerging AI tools that are reshaping the market analysis field. It encompasses detailed findings on tool features, limitations, costs, and practical use cases tailored for mid-sized marketing firms. The objective is to equip organizations with the knowledge necessary to select appropriate AI tools that can enhance their market analysis capabilities.

## 1. Identification of Top Five AI Tools

After extensive research on emerging AI technologies, the following tools have been identified as the top five for competitive market analysis:

1. **DeepSights by Market Logic**
   - **Description**: An AI assistant offering natural language responses to business intelligence inquiries, integrated with platforms like Microsoft Teams.
   - **Popularity**: Over 100,000 users globally.
   - **Source**: [Market Logic Press Release 2023](https://marketlogicsoftware.com/news/2023/deepsights-press-release/)

2. **Course5 Compete**
   - **Description**: Leverages OpenAI technology for greater market intelligence and improved product matching.
   - **Prominence**: Noted for creativity and supported by a substantial funding of $53 million.
   - **Source**: [PR Newswire Announcement 2023](https://www.prnewswire.com/news-releases/course5-intelligence-integrates-openais-gpt-models-with-their-enterprise-analytics-platforms-301777256.html)

3. **Generate by Nextatlas**
   - **Description**: Combines large language models (LLMs) with trend forecasting to deliver insights based on social media behavior.
   - **Recognition**: Positioned as leading-edge for capturing cultural shifts.
   - **Source**: [Nextatlas Blog on Generate Launch](https://blog.nextatlas.com/article/Nextalas_Generate_Trend_Prediction_Launch)

4. **Quid Predict**
   - **Description**: A generative AI tool focused on predicting trends by analyzing diverse data sources.
   - **Features**: Utilizes radar visualizations for continuous trend monitoring.
   - **Source**: [Quid Blog Release](https://www.quid.com/knowledge-hub/resource-library/blog/quid-launches-ai-product-quid-predict)

5. **Market Muse**
   - **Description**: Specializes in SEO content optimization and development of data-driven content strategies.
   - **Strengths**: Strong integration with various market intelligence tools for boosting content visibility.
   - **Source**: [Market Muse Overview](https://www.marketmuse.com)

### Evaluation of Criteria

The selection of these AI tools was governed by their prominence in the marketing industry, user reviews, integration capabilities, and effectiveness in translating data into actionable insights.

## 2. Features of Each Tool

The features of the identified tools are summarized in the following comparative table:

| **Tool**               | **Unique Features**                                      | **Enhancements for Firms**                        | **Integration Capabilities**               |
|------------------------|---------------------------------------------------------|--------------------------------------------------|---------------------------------------------|
| **DeepSights**         | Natural language processing, citation with answers.     | Quick insights save time in research.           | Microsoft Teams, API integrations.        |
| **Course5 Compete**    | AI-generated product match suggestions.                 | Tailored insights improve product launches.      | Various analytics platforms.               |
| **Generate**           | Real-time trend reporting, insights from social media.  | Rapid adaptation to consumer trends.             | REST API, integration with business tools. |
| **Quid Predict**       | Radar visualization of trends.                           | Supports proactive decision-making.              | Integrates with varied data sources.       |
| **Market Muse**        | AI-driven content strategy and optimization tools.      | Increases SEO performance and engagement.        | Content management systems.                 |

## 3. Limitations of Each Tool

The following are limitations encountered with each tool:

- **DeepSights**: Dependent on user input for accuracy; reports of slow responses during high data loads.
- **Course5 Compete**: Heavily reliant on existing datasets, which can limit exploratory analysis; slow in dynamic markets.
- **Generate**: Predictive accuracy hinges on data quality; basic understanding of social metrics suggested for efficiency.
- **Quid Predict**: Complex interface may necessitate user training; some users find filtering options too simplistic.
- **Market Muse**: Higher price point may be prohibitive for smaller firms; performance can vary with input quality.

## 4. Cost Analysis

The cost structures for these tools vary significantly, as outlined below:

| **Tool**               | **Pricing Models**                             |
|------------------------|------------------------------------------------|
| **DeepSights**         | Quarterly subscription starting from €1,000.   |
| **Course5 Compete**    | Prices based on individual quotes.            |
| **Generate**           | Free trials available; tiered pricing afterward.|
| **Quid Predict**       | Custom pricing based on usage.                 |
| **Market Muse**        | $99/month for basic features up to $499/month for enterprise services. |

## 5. Ideal Use Cases

- **DeepSights**: Best suited for organizations requiring quick insights in everyday workflows.
- **Course5 Compete**: Ideal for product launches requiring thorough competitor analysis.
- **Generate**: Most beneficial for businesses needing quick adjustments in response to market shifts.
- **Quid Predict**: Useful for ongoing predictions and strategies based on market trends.
- **Market Muse**: Optimal for companies focused on content marketing and SEO strategies.

## Conclusion

This comprehensive analysis highlights the five leading AI tools for automating competitive market analysis, detailing their features, limitations, costs, and suitable contexts for their application. By utilizing the insights gained from this report, mid-sized marketing firms can make informed decisions about which technology best fits their strategic goals and operational needs.

## Citations

- **MarTech**: [Latest AI-Powered Marketing Technology Releases](https://martech.org/a-roundup-of-the-latest-ai-powered-marketing-technology-releases/)
- **Market Logic**: [DeepSights Press Release](https://marketlogicsoftware.com/news/2023/deepsights-press-release/)
- **AlphaSense**: [Future of Generative AI in Market Research](https://www.alpha-sense.com/blog/trends/future-generative-ai-market-research/)
- **Nextatlas**: [Generate: Revolutionizing Trend Reporting with AI](https://blog.nextatlas.com/article/Nextalas_Generate_Trend_Prediction_Launch)
- **Umair Kamil**: [Top 10 AI Tools for Market Research](https://umairkamil.com/ai-tools-for-market-research/)
- **Quid**: [Quid Launches New Generative AI Prediction Tool](https://www.quid.com/knowledge-hub/resource-library/blog/quid-launches-ai-product-quid-predict)
- **QuestionPro**: [AI Tools for Business: 17 Top Picks by Function](https://www.questionpro.com/blog/ai-tools-for-business/)

This report synthesizes thorough research to provide mid-sized marketing firms with critical insights on AI tools that significantly impact their market analysis capabilities.

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**⚠️ 坑**

| 坑 | 现象 | 原因 / 对策 |
|---|---|---|
| Jupyter 里调 `crew.kickoff()` | `RuntimeError ... running event loop` | 用 `await crew.kickoff_async(...)` |
| `.env` 没有 `EXA_API_KEY` | `TypeError: str expected, not NoneType`（给 `os.environ` 赋 None） | 在 `.env` 加 key |
| description 里写成 f-string `f"...{user_query}"` | 定义时就 NameError，或提前写死 | 用普通字符串，交给 `kickoff(inputs=...)` 替换 |
| 相邻字符串字面量末尾不留空格 | 单词粘连（`eachresearch`） | 每段结尾留空格 |
| `tasks` 顺序写乱 | 写报告时还没有资料 | `tasks` 列表顺序 = 执行顺序 |

</div>

<div class="dark-orange" style="background:#2d2418; color:#fed7aa; padding:10px 24px; border-left:4px solid #fb923c; border-radius:4px; width:97%;"><style>.dark-orange strong{color:#67e8f9;}</style>

**🔁 复习自测**（先自己答，再点开看答案）

<details><summary><b>1. 这个 crew 有哪 4 个 agent、各配什么工具？</b></summary>

Research Planner（无工具）、Internet Researcher（Exa 搜索 + 网页抓取）、Fact Checker（同上）、Report Writer（无工具）。

</details>

<details><summary><b>2. <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">max_rpm</code> 和 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">max_iter</code> 分别限制什么？</b></summary>

`max_rpm`：每分钟请求数上限，防限流；`max_iter`：最多迭代轮数，到了必须给出最佳答案。

</details>

<details><summary><b>3. <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">{user_query}</code> 什么时候、被谁替换？</b></summary>

调 `kickoff(inputs={...})` 时，由 CrewAI 对 task 的 description/expected_output 和 agent 的 role/goal/backstory 做模板插值；不是 f-string。

</details>

<details><summary><b>4. 没写 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">context=</code> 时，第 3 个 task 能看到什么？</b></summary>

前两个 task 的 `.raw` 用 `\n\n----------\n\n` 拼接的文本，附在 prompt 的 "This is the context you're working with:" 后面。

</details>

<details><summary><b>5. 为什么事实核查要单独一个 agent？</b></summary>

职责分离：研究员负责「找」，核查员专门挑矛盾、错误信息和缺口，降低幻觉进入最终报告的概率。

</details>

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**：**多 agent = 把工作拆成专职角色 + 按 `tasks` 顺序接力**，前面的文本输出自动拼进后面的 prompt；`{user_query}` 由 `kickoff(inputs=...)` 填充。

</div>